In [1]:
!pip install numpy
!pip install pandas



In [2]:
import pandas as pd

df = pd.read_excel('/content/road_accident_imu_dataset_8000.xlsx')
df.head()

,Timestamp,Acc_X,Acc_Y,Acc_Z,Gyro_X,Gyro_Y,Gyro_Z,Speed_kmh
0,2026-01-01 10:00:00,0.448357,0.083487,9.963792,-0.013569,0.056601,-0.003593,31.601007
1,2026-01-01 10:00:01,0.130868,-0.151825,9.711793,-0.015690,-0.060578,0.070635,65.337499
2,2026-01-01 10:00:02,0.523844,0.013813,9.426302,0.091270,-0.013817,0.010091,31.599463
3,2026-01-01 10:00:03,0.961515,0.457366,9.561250,0.113221,-0.048402,0.024528,27.447587
4,2026-01-01 10:00:04,0.082923,0.738928,9.802856,-0.026444,0.029465,0.055870,79.673292


In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# --- 1. LOAD YOUR DATASET IN COLAB ---
# NOTE: Make sure 'road_accident_imu_dataset_8000.xlsx' is uploaded to your Colab /content/ folder!
print("Loading dataset...")
df = pd.read_excel('road_accident_imu_dataset_8000.xlsx')

# --- 2. PREPROCESS AS PER YOUR NOTEBOOK ---
def add_physics_features(df):
    df = df.copy()
    df["acc_mag"] = np.sqrt(df["Acc_X"]**2 + df["Acc_Y"]**2 + df["Acc_Z"]**2)
    df["gyro_mag"] = np.sqrt(df["Gyro_X"]**2 + df["Gyro_Y"]**2 + df["Gyro_Z"]**2)
    df["jerk"] = df["acc_mag"].diff().fillna(0)
    return df

df = add_physics_features(df)

# FIX: Create a placeholder 'label' column.
# The model expects 5 classes (0-4), so we'll assign random integers for now.
# You MUST replace this with actual, meaningful labels for your accident detection task.
# For example, you might define an accident based on thresholds of 'jerk' or other features,
# or load a dataset that already contains these labels.
df['label'] = np.random.randint(0, 5, size=len(df))

feature_cols = ["Acc_X", "Acc_Y", "Acc_Z", "Gyro_X", "Gyro_Y", "Gyro_Z", "Speed_kmh", "acc_mag", "gyro_mag", "jerk"]
X = df[feature_cols].to_numpy()
y = df["label"].to_numpy()

# --- 3. FIT THE SCALER ---
scaler = StandardScaler()
X = scaler.fit_transform(X)

# --- IMPORTANT: EXPORT SCALER VALUES FOR ANDROID KOTLIN ---
print("\n" + "="*50)
print("👉 COPY THESE VALUES FOR ANDROID KOTLIN INTEGRATION:")
print(f"val SCALER_MEANS = floatArrayOf({', '.join(map(lambda x: str(x) + 'f', scaler.mean_))})")
print(f"val SCALER_SCALES = floatArrayOf({', '.join(map(lambda x: str(x) + 'f', scaler.scale_))})")
print("="*50 + "\n")

# --- 4. CREATE TIME WINDOWS (Size=20) ---
def create_windows(X, y, window_size=20):
    X_windows, y_windows = [], []
    for i in range(len(X) - window_size):
        X_windows.append(X[i:i + window_size])
        y_windows.append(y[i + window_size])
    return np.array(X_windows), np.array(y_windows)

WINDOW_SIZE = 20
X_seq, y_seq = create_windows(X, y, WINDOW_SIZE)

# Ensure everything is Float32 for TFLite compatibility!
X_seq = X_seq.astype(np.float32)

from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
y_seq_cat = to_categorical(y_seq, 5)

X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq_cat, test_size=0.2, random_state=42, stratify=y_seq)

# --- 5. BUILD AND TRAIN THE 1D CNN MODEL ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, Input

print("Training the CNN Model...")
model = Sequential([
    Input(shape=(WINDOW_SIZE, X_train.shape[2])), # Explicitly define input shape
    Conv1D(64, 3, activation="relu"),
    MaxPooling1D(2),
    Conv1D(128, 3, activation="relu"),
    MaxPooling1D(2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(5, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1)

# --- 6. EXPORT TO TF-LITE (.tflite) ---
print("\nExporting to TensorFlow Lite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimizations for mobile execution
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

tflite_model = converter.convert()

with open('accident_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ 'accident_model.tflite' has been generated successfully!")
print("Download it from the Colab file browser (left sidebar) and upload it here, so I can put it into the Android project.")

Loading dataset...

👉 COPY THESE VALUES FOR ANDROID KOTLIN INTEGRATION:
val SCALER_MEANS = floatArrayOf(0.5712545009774022f, 0.47822156115363346f, 10.07537117174066f, 0.01969896938665963f, 0.01948064282662192f, 0.020123306585414745f, 45.07206561487641f, 10.20014427320242f, 0.0857426030104364f, 0.00041608147070049806f)
val SCALER_SCALES = floatArrayOf(1.1308369487637457f, 1.1277571353889624f, 1.0527783836052091f, 0.04971329575020809f, 0.05021980524541636f, 0.049868251212801196f, 20.98580444705255f, 1.2977516352269454f, 0.036069120158641534f, 0.521029508367747f)

Training the CNN Model...
Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2080 - loss: 1.6260
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2188 - loss: 1.6092
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2398 - loss: 1.6027
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2437 - loss: 1.5980
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy